In [1]:
import codecs, json, re
from random import shuffle

# 数据说明
原始数据地址：https://nijianmo.github.io/amazon/index.html

In [ ]:
第一步：读取 video game 信息

In [2]:
# key 是 productID，value是 title
games = {}
cc = 0

In [3]:
with codecs.open('./data/src_data/meta_Video_Games.json', mode='r') as fin:
    for line in fin:
        tmp_info = json.loads(line.strip())
        # asin - ID of the product
        # title - name of the product
        games[tmp_info["asin"]] = tmp_info["title"]
        if len(games) % 10000 == 0:
            print(f'Length of games: {len(games)}')

Length of games: 10000
Length of games: 20000
Length of games: 30000
Length of games: 40000
Length of games: 50000
Length of games: 60000
Length of games: 70000


第二步：读取用户评分信息

In [4]:
# key 是 userid，value 是评价的游戏和评分
user_reviews = {}

In [5]:
cc = 0
with codecs.open('./data/src_data/Video_Games_5.json', mode='r') as fin:
    for line in fin:
        tmp_info = json.loads(line.strip())
        
        # reviewerID - ID of the reviewer
        reviewer_id = tmp_info["reviewerID"]
        
        time_info = re.split(', | ', tmp_info["reviewTime"])
        review_time = time_info[2] + '-' + time_info[0] + '-' + time_info[1]
        
        # asin - ID of the product
        product_id = tmp_info["asin"]
        
        # overall - rating of the product
        rating = tmp_info["overall"]
        
        # if cc > 1000:
        #     break
        
        # print(tmp_info)
        # print(user_reviews)
        
        if product_id in games.keys():
            product_title = games[product_id]
        
            if reviewer_id in user_reviews.keys():
                user_reviews[reviewer_id].append((product_title, rating, review_time))
            else:
                user_reviews[reviewer_id] = [(product_title, rating, review_time)]
                
        if len(user_reviews) % 10000 == 0:
            print(f'Length of user_reviews: {len(user_reviews)}')
        
        cc += 1

Length of user_reviews: 10000
Length of user_reviews: 20000
Length of user_reviews: 20000
Length of user_reviews: 20000
Length of user_reviews: 20000
Length of user_reviews: 20000
Length of user_reviews: 30000
Length of user_reviews: 40000
Length of user_reviews: 40000
Length of user_reviews: 40000
Length of user_reviews: 40000
Length of user_reviews: 40000
Length of user_reviews: 40000
Length of user_reviews: 50000
Length of user_reviews: 50000
Length of user_reviews: 50000
Length of user_reviews: 50000
Length of user_reviews: 50000


In [6]:
user_reviews_sorted = {}
for k, v in user_reviews.items():
    # 首先去重
    v = list(set(v))
    # 然后根据评价时间从小到大排序，表示用户的评价历史
    v_sorted = sorted(v, key=lambda x: x[2])
    # 选择具有7个及以上的评论样本
    if len(v) >= 7:
        # print(f'v: {v}, v_sorted: {v_sorted}')
        user_reviews_sorted[k] = v_sorted
print(f'Length of user_reviews_sorted: {len(user_reviews_sorted)}')

Length of user_reviews_sorted: 25346


# 训练数据生成

In [7]:
# 总样本
samples = []
# 指令
instruction = "You are an assistant working on Video Games recommendations. Given the user's history of Video Games they have shopped, which includes the \"Title\" of the Video Games and the \"Rating\" the user rate (the Rating value is like or dislike), please decide whether the user likes to shop the target Video Games by outputting the order of their titles."

In [8]:
samples = []
cc = 0
for k, v in user_reviews_sorted.items():
    # print('-'*10)
    # print(v)
    sample_input = "User shopped Video Games histories (Title and Rating): \n"
    # 前面的当作对话历史
    for vv in v[0: -2]:
        # 当 rating 大于 3.0 的时候设置为 like
        if vv[1] > 3.0:
            rating = 'like'
        # 当 rating 小于等于 3.0 的时候设置为 dislike
        else:
            rating = 'dislike'
        sample_input += "<Title: {}, Rating: {}>\n".format(vv[0], rating)
    
    sample_input += "Based on the Video Games histories, please sort the following two Video Games titles. The one in the front is what the user like and should be recommended to user: \n"
    
    # 最后两个设置为需要预测的目标
    sample_input += "<Title: " + v[-2][0] + '>\n'
    sample_input += "<Title: " + v[-1][0] + '>\n'
    
    # print(f'v[-1][1]: {v[-1][1]}, v[-2][1]: {v[-2][1]}')
    # 保证有一个是 like，有一个是 dislike
    if (v[-1][1] > 3.0 and v[-2][1] <= 3.0) or (v[-1][1] <= 3.0 and v[-2][1] > 3.0):
        # print(f'v[-1][1] != v[-2][1]: {v[-1][1]}, {v[-2][1]}')
        if v[-1][1] > v[-2][1]:
            # like
            option1 = v[-1][0]
            # dislike
            option2 = v[-2][0]
        else:
            # like
            option1 = v[-2][0]
            # dislike
            option2 = v[-1][0]
        
        # chosen 是 like 在前面
        chosen = "<Title: " + option1 + '>\n' + "<Title: " + option2 + '>'
        # rejected 是 dislike 在前面
        rejected = "<Title: " + option2 + '>\n' + "<Title: " + option1 + '>'

        sample = {
            "instruction": instruction,
            "input": sample_input,
            "chosen": chosen,
            "rejected": rejected
        }
        # print(f'--------')
        # print(v)
        # print(sample)
        samples.append(sample)

        if len(samples) % 10000 == 0:
            print(f'Length of samples: {len(samples)}')
    
    # cc += 1
    # if cc > 10:
    #     break
        
print(f'Length of samples: {len(samples)}')

Length of samples: 6430


In [9]:
# 查看一下样本
samples[0:10]

[{'instruction': 'You are an assistant working on Video Games recommendations. Given the user\'s history of Video Games they have shopped, which includes the "Title" of the Video Games and the "Rating" the user rate (the Rating value is like or dislike), please decide whether the user likes to shop the target Video Games by outputting the order of their titles.',
  'input': "User shopped Video Games histories (Title and Rating): \n<Title: Halo 2 - Xbox, Rating: like>\n<Title: Kohan II: Kings of War - PC, Rating: dislike>\n<Title: The Lord of the Rings: The Battle for Middle-Earth, Rating: like>\n<Title: Warhammer 40,000 Dawn of War Game of the Year - PC, Rating: dislike>\n<Title: Sid Meier's Alpha Centauri - PC, Rating: like>\n<Title: Sid Meier's Civilization IV, Rating: dislike>\n<Title: Galactic Civilizations 2: Dread Lords - PC, Rating: like>\n<Title: Battle Realms - PC, Rating: like>\n<Title: The Darkness, Rating: dislike>\n<Title: Halo 3 - Xbox 360, Rating: dislike>\n<Title: Shatt

In [10]:
# 第三步 划分 train 和 test 保存样本
# 首先打乱
shuffle(samples)

train = samples[:int(len(samples)*0.8)]
test = samples[int(len(samples)*0.8):]

print(f'总样本数: {len(samples)}，训练集样本数: {len(train)}，测试集样本数: {len(test)}')

with open("./data/processed/rlhf_train.json", "w", encoding='utf-8') as save_file:
    json.dump(train, save_file, indent=4)
    
with open("./data/processed/rlhf_test.json", "w", encoding='utf-8') as save_file:
    json.dump(test, save_file, indent=4) # , sort_keys=True

总样本数: 6430，训练集样本数: 5144，测试集样本数: 1286
